# Bonus: SQL (adaptado a lab local)

**Qué se ve aquí:** un agente que consulta una base SQLite (`resources/Chinook.db`, tienda de música) escribiendo SQL con una tool.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [9]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest", num_ctx=16384)   # el esquema de la BD + resultados SQL ocupan contexto

In [10]:
# Sin cambios
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [11]:
# Sin cambios
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[sql_query]
# )

In [13]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=[sql_query]
)

In [14]:
# Sin cambios
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [15]:
# Sin cambios
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='77abd5a0-542f-4a3c-93e0-4f3606573e72'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T04:25:30.6567427Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2602430000, 'load_duration': 8416400, 'prompt_eval_count': 78, 'prompt_eval_duration': 284140000, 'eval_count': 38, 'eval_duration': 2299213000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10a4f-3e7d-70e3-a637-5585eed15780-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT artist_name FROM artists WHERE artist_name LIKE 'S%' ORDER BY popularity DESC LIMIT 1"}, 'id': '8dda6a50-f05c-4cd9-9f6e-612d9c36cf84', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 38, 'total_tokens': 116}),
 ToolMessage(content="Error: (s

In [16]:
# Sin cambios
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT artist_name FROM artists WHERE artist_name LIKE 'S%' ORDER BY popularity DESC LIMIT 1


## 🟢 2º intento — el agente debe descubrir el esquema

**Qué pasó en el 1er intento:** sin system prompt, gemma4 **adivinó** el esquema (`artists`, `artist_name`, `popularity`), recibió `no such table: artists` y **le preguntó al usuario** el esquema en vez de explorarlo.

**Corrección:**
1. System prompt: descubrir las tablas y columnas antes de consultar, y no preguntar el esquema.
2. Definir qué significa *popular* (la pregunta es ambigua): aquí, **canciones vendidas** (`InvoiceLine.Quantity`).

**Referencia (calculada aparte contra la BD):** por canciones vendidas, **Smashing Pumpkins (24)**; luego Spyro Gyra (20) y Soundgarden (15). Por número de canciones en el catálogo también gana Smashing Pumpkins (34).

In [18]:
# 🟢 ADAPTADO LOCAL — 2º intento: system prompt que obliga a explorar el esquema
# Motivo: sin instrucciones, gemma4 inventó la tabla `artists` y terminó preguntando al usuario.
agent_v2 = create_agent(
    model=MODEL,
    tools=[sql_query],
    system_prompt="""
    You answer questions about a SQLite database using the sql_query tool.
    1. ALWAYS start by listing tables: SELECT name FROM sqlite_master WHERE type='table'
    2. Then inspect the columns you need: PRAGMA table_info(<table>)
    3. Only then write the query. If a query fails, read the error, fix it and retry.
    Never ask the user for the schema. "Popular" means most tracks sold (SUM of InvoiceLine.Quantity).
    Show the final SQL you used and the answer.
    """
)

response_v2 = agent_v2.invoke({"messages": [question]}, config={"tags": ["SQL-v2"]})
for m in response_v2["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print("SQL ->", tc["args"].get("query"))
print("\nRESPUESTA:", response_v2["messages"][-1].content)

SQL -> SELECT name FROM sqlite_master WHERE type='table'
SQL -> PRAGMA table_info(Artist)
SQL -> PRAGMA table_info(InvoiceLine)
SQL -> SELECT T1.Name FROM Artist AS T1 INNER JOIN InvoiceLine AS T2 ON T1.ArtistId = (SELECT ArtistId FROM Track WHERE TrackId = T2.TrackId) WHERE T1.Name LIKE 'S%' GROUP BY T1.Name ORDER BY SUM(T2.Quantity) DESC LIMIT 1

RESPUESTA: The most popular artist beginning with 'S' is System Of A Down.


In [19]:
# Sin cambios
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT artist_name FROM artists WHERE artist_name LIKE 'S%' ORDER BY popularity DESC LIMIT 1


## 🟢 3er intento — esquema real en el prompt + evaluación ✅

**Qué falló en el 2º:** exploró solo `Artist` e `InvoiceLine`, unió por `Track.ArtistId` (no existe) y SQLite lo resolvió contra la tabla externa → todos empataron en 2240 → *System Of A Down* (falso).

**Corrección:** `db.get_table_info([...])` mete en el prompt el esquema real de las 4 tablas del camino y se le da el camino de JOIN `InvoiceLine → Track → Album → Artist`.

**Resultado:** SQL correcto con los 3 JOIN → **Smashing Pumpkins** ✅. La celda de evaluación lo compara contra la referencia: v2 ❌, v3 ✅.

In [21]:
# 🟢 ADAPTADO LOCAL — 3er intento: esquema real en el prompt + camino de JOIN
# Motivo: en el 2º intento revisó solo Artist e InvoiceLine, usó Track.ArtistId (no existe)
#         y SQLite no dio error → todos empataron en 2240 → respuesta falsa (System Of A Down).
@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

schema = db.get_table_info(["Artist", "Album", "Track", "InvoiceLine"])

agent_v3 = create_agent(
    model=MODEL,
    tools=[sql_query],
    system_prompt=f"""
    You answer questions about a SQLite music store database using the sql_query tool.
    This is the REAL schema (use only these columns):
    {schema}

    Join path from a sale to its artist:
    InvoiceLine.TrackId -> Track.TrackId, Track.AlbumId -> Album.AlbumId, Album.ArtistId -> Artist.ArtistId
    "Popular" means most tracks sold: SUM(InvoiceLine.Quantity).
    Never ask the user for the schema. Show the final SQL you used and the number behind the answer.
    """
)

response_v3 = agent_v3.invoke({"messages": [question]}, config={"tags": ["SQL-v3"]})
for m in response_v3["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print("SQL ->", tc["args"].get("query"))
print("\nRESPUESTA:", response_v3["messages"][-1].content)

SQL -> SELECT T1.Name FROM Artist AS T1 JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId JOIN Track AS T3 ON T2.AlbumId = T3.AlbumId JOIN InvoiceLine AS T4 ON T3.TrackId = T4.TrackId WHERE T1.Name LIKE 'S%' GROUP BY T1.ArtistId ORDER BY SUM(T4.Quantity) DESC LIMIT 1

RESPUESTA: The most popular artist beginning with 'S' is Smashing Pumpkins. 1


In [22]:
# 🟢 ADAPTADO LOCAL — Evaluación: comparar contra una referencia calculada aparte (dominio Test)
referencia = db.run("""
SELECT ar.Name, SUM(il.Quantity) AS vendidas
FROM InvoiceLine il
JOIN Track t  ON il.TrackId = t.TrackId
JOIN Album al ON t.AlbumId = al.AlbumId
JOIN Artist ar ON al.ArtistId = ar.ArtistId
WHERE ar.Name LIKE 'S%'
GROUP BY ar.Name ORDER BY vendidas DESC LIMIT 3
""")
print("Referencia (top 3):", referencia)

esperado = "Smashing Pumpkins"
for nombre, r in [("v2", response_v2), ("v3", response_v3)]:
    ok = esperado.lower() in r["messages"][-1].content.lower()
    print(f"{nombre}: {'✅ correcto' if ok else '❌ incorrecto'}")

Referencia (top 3): [('Smashing Pumpkins', 24), ('Spyro Gyra', 20), ('Soundgarden', 15)]
v2: ❌ incorrecto
v3: ✅ correcto
